# Chronic Kidney Disease (CKD) Prediction — Machine Learning Project

**Goal:** Build and evaluate classification models to predict CKD from the cleaned dataset.

**Dataset file:** `kidney_cleaned.csv` (put it in the same folder as this notebook).

**Important methodological notes**
- `Patient_ID` is an identifier and is excluded from model training.
- `Age_Group`, `Blood_Pressure_Category`, and `Blood_Glucose_Category` are derived from existing numeric variables, so they are excluded to avoid redundant representations.
- Missing-value handling, encoding, and scaling are fitted inside each training fold through a Pipeline.
- Outliers are retained; statistical outliers are not automatically data errors, especially in medical measurements.
- The source, sampling method, measurement units, and target-label creation process are not documented in the provided file. Results are therefore academic and exploratory, not clinical validation.


## 1. Install / import libraries
If a library is missing, run the install command in this notebook (not in PowerShell).

In [ ]:
%pip install -q -U pandas numpy scikit-learn matplotlib seaborn joblib

In [ ]:
import warnings
warnings.filterwarnings('ignore')

import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import joblib

from pathlib import Path
from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_score
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, ExtraTreesClassifier
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score, roc_auc_score,
    classification_report, confusion_matrix, ConfusionMatrixDisplay,
    RocCurveDisplay
)
from sklearn.base import clone

RANDOM_STATE = 42
pd.set_option('display.max_columns', None)
sns.set_theme(style='whitegrid')

## 2. Load the cleaned dataset

In [ ]:
DATA_PATH = Path('kidney_cleaned.csv')

if not DATA_PATH.exists():
    raise FileNotFoundError(
        "kidney_cleaned.csv was not found. Put the CSV file in the same folder as this notebook."
    )

df = pd.read_csv(DATA_PATH)
print('Dataset shape:', df.shape)
display(df.head())

## 3. Data checks and target preparation

In [ ]:
TARGET = 'Chronic_Kidney_Disease'

if TARGET not in df.columns:
    raise ValueError(f'Target column {TARGET!r} is missing from the dataset.')

print('Missing values in target:', df[TARGET].isna().sum())
print('Original target values:')
display(df[TARGET].value_counts(dropna=False))

# Remove rows only when the target itself is missing.
df = df.dropna(subset=[TARGET]).copy()

# Map the dataset labels: si = CKD (1), no = No CKD (0).
target_text = df[TARGET].astype(str).str.strip().str.lower()
y = target_text.map({'si': 1, 'yes': 1, 'no': 0})

if y.isna().any():
    unknown = sorted(target_text[y.isna()].unique().tolist())
    raise ValueError(f'Unexpected target labels found: {unknown}')

print('Target distribution (0 = No CKD, 1 = CKD):')
display(y.value_counts().rename(index={0: 'No CKD', 1: 'CKD'}))
print('Target percentage:')
display((y.value_counts(normalize=True) * 100).round(2))

## 4. Select features
We exclude the patient identifier and the three derived category columns. The original clinical measurements are retained.

In [ ]:
EXCLUDED_COLUMNS = [
    TARGET,
    'Patient_ID',
    'Age_Group',
    'Blood_Pressure_Category',
    'Blood_Glucose_Category'
]

X = df.drop(columns=[c for c in EXCLUDED_COLUMNS if c in df.columns]).copy()

# Normalize text values and whitespace in categorical columns.
for col in X.select_dtypes(include=['object', 'category']).columns:
    X[col] = X[col].astype('string').str.strip().str.lower()

numeric_features = X.select_dtypes(include=np.number).columns.tolist()
categorical_features = X.select_dtypes(exclude=np.number).columns.tolist()

print('Number of features:', X.shape[1])
print('Numeric features:', numeric_features)
print('Categorical features:', categorical_features)
print('Missing values in X:', int(X.isna().sum().sum()))

## 5. Split data into training and test sets
The test set is held out and is not used for cross-validation or model selection.

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.20,
    random_state=RANDOM_STATE,
    stratify=y
)

print('Training set:', X_train.shape)
print('Test set:', X_test.shape)
print('Training target ratio:')
display(y_train.value_counts(normalize=True).round(3))

## 6. Preprocessing pipeline
Numeric values are median-imputed and standardized. Categorical values are mode-imputed and one-hot encoded. These steps are learned only from the training data in each CV fold.

In [ ]:
numeric_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', StandardScaler())
])

categorical_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('onehot', OneHotEncoder(handle_unknown='ignore'))
])

preprocessor = ColumnTransformer(transformers=[
    ('num', numeric_transformer, numeric_features),
    ('cat', categorical_transformer, categorical_features)
])

models = {
    'Logistic Regression': LogisticRegression(max_iter=2000, class_weight='balanced'),
    'Random Forest': RandomForestClassifier(
        n_estimators=250, random_state=RANDOM_STATE,
        class_weight='balanced_subsample', n_jobs=-1
    ),
    'Extra Trees': ExtraTreesClassifier(
        n_estimators=250, random_state=RANDOM_STATE,
        class_weight='balanced', n_jobs=-1
    )
}

cv = StratifiedKFold(n_splits=3, shuffle=True, random_state=RANDOM_STATE)
print('Preprocessing and models are ready.')

## 7. Train and compare models using 3-fold cross-validation
Model selection is based on mean training-fold F1-score. The held-out test set remains untouched until the final evaluation.

In [ ]:
cv_results = []

for name, estimator in models.items():
    pipe = Pipeline(steps=[('preprocessor', clone(preprocessor)), ('model', clone(estimator))])
    scores = cross_val_score(
        pipe, X_train, y_train,
        cv=cv, scoring='f1', n_jobs=1
    )
    cv_results.append({
        'Model': name,
        'CV F1 Mean': scores.mean(),
        'CV F1 Std': scores.std()
    })
    print(f'{name}: F1 = {scores.mean():.4f} ± {scores.std():.4f}')

cv_results_df = pd.DataFrame(cv_results).sort_values('CV F1 Mean', ascending=False)
display(cv_results_df)
best_name = cv_results_df.iloc[0]['Model']
print('Selected model based on CV F1:', best_name)

## 8. Fit the selected model and evaluate on the test set

In [ ]:
best_pipeline = Pipeline(steps=[
    ('preprocessor', clone(preprocessor)),
    ('model', clone(models[best_name]))
])
best_pipeline.fit(X_train, y_train)

y_pred = best_pipeline.predict(X_test)
y_prob = best_pipeline.predict_proba(X_test)[:, 1]

final_metrics = {
    'Model': best_name,
    'Accuracy': accuracy_score(y_test, y_pred),
    'Precision': precision_score(y_test, y_pred, zero_division=0),
    'Recall': recall_score(y_test, y_pred, zero_division=0),
    'F1-Score': f1_score(y_test, y_pred, zero_division=0),
    'ROC-AUC': roc_auc_score(y_test, y_prob)
}

final_metrics_df = pd.DataFrame([final_metrics])
display(final_metrics_df.round(4))
print(classification_report(y_test, y_pred, target_names=['No CKD', 'CKD'], zero_division=0))

## 9. Confusion Matrix and ROC Curve

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

ConfusionMatrixDisplay.from_predictions(
    y_test, y_pred,
    display_labels=['No CKD', 'CKD'],
    cmap='Blues',
    ax=axes[0],
    colorbar=False
)
axes[0].set_title(f'Confusion Matrix — {best_name}')

RocCurveDisplay.from_predictions(y_test, y_prob, ax=axes[1])
axes[1].plot([0, 1], [0, 1], linestyle='--', label='Random Classifier')
axes[1].set_title(f'ROC Curve — {best_name}')
axes[1].legend()

plt.tight_layout()
plt.savefig('ckd_evaluation_plots.png', dpi=300, bbox_inches='tight')
plt.show()

## 10. Sensitivity check: train without Red_Blood_Cells and Pus_Cells
These two variables showed unusually strong associations with the target in the exploratory analysis. This experiment checks how much the selected model's performance changes when they are excluded. It does **not** prove leakage or prove that the remaining data are unbiased.

In [ ]:
suspicious_features = ['Red_Blood_Cells', 'Pus_Cells']
reduced_features = [c for c in X.columns if c not in suspicious_features]

if len(reduced_features) == len(X.columns):
    print('Sensitivity check skipped: the two selected columns were not found.')
else:
    X_train_reduced = X_train[reduced_features].copy()
    X_test_reduced = X_test[reduced_features].copy()

    reduced_num = X_train_reduced.select_dtypes(include=np.number).columns.tolist()
    reduced_cat = X_train_reduced.select_dtypes(exclude=np.number).columns.tolist()

    reduced_preprocessor = ColumnTransformer(transformers=[
        ('num', Pipeline(steps=[
            ('imputer', SimpleImputer(strategy='median')),
            ('scaler', StandardScaler())
        ]), reduced_num),
        ('cat', Pipeline(steps=[
            ('imputer', SimpleImputer(strategy='most_frequent')),
            ('onehot', OneHotEncoder(handle_unknown='ignore'))
        ]), reduced_cat)
    ])

    sensitivity_model = Pipeline(steps=[
        ('preprocessor', reduced_preprocessor),
        ('model', clone(models[best_name]))
    ])
    sensitivity_model.fit(X_train_reduced, y_train)
    reduced_pred = sensitivity_model.predict(X_test_reduced)
    reduced_prob = sensitivity_model.predict_proba(X_test_reduced)[:, 1]

    sensitivity_results = pd.DataFrame([
        {
            'Experiment': 'All selected features',
            'Accuracy': accuracy_score(y_test, y_pred),
            'F1-Score': f1_score(y_test, y_pred, zero_division=0),
            'ROC-AUC': roc_auc_score(y_test, y_prob)
        },
        {
            'Experiment': 'Without Red_Blood_Cells and Pus_Cells',
            'Accuracy': accuracy_score(y_test, reduced_pred),
            'F1-Score': f1_score(y_test, reduced_pred, zero_division=0),
            'ROC-AUC': roc_auc_score(y_test, reduced_prob)
        }
    ])
    display(sensitivity_results.round(4))

## 11. Save the trained model and results

In [ ]:
# Save the full preprocessing + model pipeline. Use this same pipeline for future predictions.
joblib.dump(best_pipeline, 'ckd_best_model.joblib')

cv_results_df.to_csv('ckd_cv_model_comparison.csv', index=False)
final_metrics_df.to_csv('ckd_final_test_metrics.csv', index=False)

predictions_df = pd.DataFrame({
    'Actual': y_test.map({0: 'No CKD', 1: 'CKD'}).values,
    'Predicted': pd.Series(y_pred, index=y_test.index).map({0: 'No CKD', 1: 'CKD'}).values,
    'Probability_CKD': y_prob
}, index=y_test.index)
predictions_df.to_csv('ckd_test_predictions.csv', index_label='Original_Row_Index')

with open('ckd_model_features.json', 'w', encoding='utf-8') as f:
    json.dump(X.columns.tolist(), f, ensure_ascii=False, indent=2)

print('Saved files:')
for filename in [
    'ckd_best_model.joblib',
    'ckd_cv_model_comparison.csv',
    'ckd_final_test_metrics.csv',
    'ckd_test_predictions.csv',
    'ckd_model_features.json',
    'ckd_evaluation_plots.png'
]:
    print('-', filename)

## 12. Example: load the saved model and predict new records
The input DataFrame must contain the same feature columns used during training. Do not include `Patient_ID`, the target, or the derived category columns.

```python
loaded_model = joblib.load('ckd_best_model.joblib')
new_patients = X_test.head(3).copy()  # Demonstration only
predictions = loaded_model.predict(new_patients)
probabilities = loaded_model.predict_proba(new_patients)[:, 1]

pd.DataFrame({
    'Prediction': pd.Series(predictions).map({0: 'No CKD', 1: 'CKD'}),
    'CKD_Probability': probabilities
})
```

## 13. Conclusion and limitations

Use the measured test metrics from this notebook in the project report; do not copy expected or previously observed scores.

**Limitations to report:**
1. The dataset file does not document its original source, sampling procedure, measurement units, or how the target labels were assigned.
2. Very strong associations between some categorical variables and the target require independent verification. The sensitivity experiment helps describe their influence but cannot establish the cause.
3. A random train/test split does not replace external validation on an independent dataset or validation across hospitals and populations.
4. This is an educational machine-learning project, not a clinically validated diagnostic system. It must not be used to diagnose or make treatment decisions.
